In [10]:
# Install library yang diperlukan (jalankan sekali)
#%pip install fuzzywuzzy python-Levenshtein pandas

import pandas as pd
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import warnings
warnings.filterwarnings('ignore')

print("Library berhasil diimport!")

Library berhasil diimport!


In [11]:
# Load data dari ketiga CSV
peserta_mcf_df = pd.read_csv('peserta mcf.csv')
peserta_mos_df = pd.read_csv('peserta mos.csv')
certiport_df = pd.read_csv('certiport.csv', skiprows=4)  # Skip 4 baris header kosong

# Bersihkan kolom certiport (hapus kolom kosong)
certiport_df = certiport_df.dropna(axis=1, how='all')

# Tambahkan kolom untuk membedakan sumber data
peserta_mcf_df['Sumber'] = 'MCF'
peserta_mos_df['Sumber'] = 'MOS'

# Gabungkan kedua data peserta
peserta_df = pd.concat([peserta_mcf_df, peserta_mos_df], ignore_index=True)

# Filter data certiport berdasarkan Exam
# MCF: AI-900: Microsoft Azure AI Fundamentals
# MOS: Yang mengandung "Office 2019" (Word, Excel, PowerPoint)
certiport_mcf = certiport_df[certiport_df['Exam'].str.contains('AI-900', na=False)].copy()
certiport_mos = certiport_df[certiport_df['Exam'].str.contains('Office 2019', na=False)].copy()

# Tambahkan kolom sumber untuk certiport
certiport_mcf['Sumber_Certiport'] = 'MCF'
certiport_mos['Sumber_Certiport'] = 'MOS'

print(f"=" * 70)
print("DATA PESERTA PENDAFTAR")
print(f"=" * 70)
print(f"Jumlah peserta MCF (Azure AI-900): {len(peserta_mcf_df)}")
print(f"Jumlah peserta MOS (Office 2019): {len(peserta_mos_df)}")
print(f"Total peserta: {len(peserta_df)}")

print(f"\n{'=' * 70}")
print("DATA CERTIPORT (HASIL UJIAN)")
print(f"=" * 70)
print(f"Total data di Certiport: {len(certiport_df)}")
print(f"Data MCF (AI-900): {len(certiport_mcf)}")
print(f"Data MOS (Office 2019): {len(certiport_mos)}")

print(f"\n--- Exam yang tersedia di Certiport ---")
print(certiport_df['Exam'].value_counts())

print(f"\n--- Preview Data Certiport MCF ---")
print(certiport_mcf[['First Name', 'Last Name', 'Exam', 'Result']].head())
print(f"\n--- Preview Data Certiport MOS ---")
print(certiport_mos[['First Name', 'Last Name', 'Exam', 'Result']].head())

DATA PESERTA PENDAFTAR
Jumlah peserta MCF (Azure AI-900): 56
Jumlah peserta MOS (Office 2019): 101
Total peserta: 157

DATA CERTIPORT (HASIL UJIAN)
Total data di Certiport: 4853
Data MCF (AI-900): 897
Data MOS (Office 2019): 3445

--- Exam yang tersedia di Certiport ---
Exam
Microsoft Word (Office 2019)                                         2714
AI-900: Microsoft Azure AI Fundamentals                               897
Microsoft Excel (Office 2019)                                         624
Microsoft Word (Office 2016)                                          287
Microsoft Excel (Office 2016)                                         161
Microsoft PowerPoint (Office 2019)                                    104
Microsoft PowerPoint (Office 2016)                                     53
Microsoft Word Expert (Office 2019)                                     3
Microsoft Excel (Microsoft 365 Apps)                                    2
Microsoft Word (Microsoft 365 Apps)                       

In [12]:
# Fungsi untuk normalisasi nama (lowercase, hapus spasi berlebih)
def normalize_name(name):
    if pd.isna(name):
        return ""
    # Hapus karakter khusus kecuali huruf, angka, spasi, dan titik
    cleaned = ''.join(c for c in str(name) if c.isalnum() or c.isspace() or c == '.')
    return ' '.join(cleaned.upper().strip().split())

# Fungsi untuk membalik urutan nama (antisipasi nama terbalik)
def reverse_name(name):
    parts = name.split()
    if len(parts) >= 2:
        # Coba balik first name dan last name
        return ' '.join(parts[::-1])
    return name

# Fungsi untuk menghitung jumlah kata yang cocok
def count_matching_words(name1, name2, min_word_length=2):
    """
    Menghitung jumlah kata yang sama antara dua nama.
    Hanya menghitung kata dengan panjang minimal min_word_length.
    """
    words1 = set(w for w in name1.split() if len(w) >= min_word_length)
    words2 = set(w for w in name2.split() if len(w) >= min_word_length)
    return len(words1.intersection(words2))

# Fungsi fuzzy matching dengan berbagai strategi + filter minimal 2 kata cocok
def find_best_match(peserta_name, certiport_names, threshold=80, min_matching_words=2):
    """
    Mencari kecocokan terbaik dengan berbagai strategi:
    1. Exact match
    2. Fuzzy match pada full name
    3. Fuzzy match pada nama terbalik
    4. Token set ratio (mengabaikan urutan kata)
    
    PENTING: Minimal harus ada 2 kata yang cocok untuk dianggap match
    """
    peserta_normalized = normalize_name(peserta_name)
    
    # 1. Exact match
    if peserta_normalized in certiport_names:
        return peserta_normalized, 100, "Exact Match"
    
    # 2. Cek nama terbalik exact
    reversed_name = reverse_name(peserta_normalized)
    if reversed_name in certiport_names:
        return reversed_name, 100, "Exact Match (Nama Terbalik)"
    
    best_score = 0
    best_match = None
    match_type = ""
    best_word_count = 0
    
    for cert_name in certiport_names:
        # Hitung jumlah kata yang cocok
        matching_words = count_matching_words(peserta_normalized, cert_name)
        
        # Skip jika kata yang cocok kurang dari minimum
        if matching_words < min_matching_words:
            continue
        
        # Fuzzy ratio normal
        score1 = fuzz.ratio(peserta_normalized, cert_name)
        # Token set ratio (tidak peduli urutan kata)
        score2 = fuzz.token_set_ratio(peserta_normalized, cert_name)
        # Token sort ratio
        score3 = fuzz.token_sort_ratio(peserta_normalized, cert_name)
        # Partial ratio
        score4 = fuzz.partial_ratio(peserta_normalized, cert_name)
        
        # Ambil skor tertinggi
        max_score = max(score1, score2, score3, score4)
        
        # Prioritaskan yang punya lebih banyak kata cocok, lalu skor tertinggi
        if matching_words > best_word_count or (matching_words == best_word_count and max_score > best_score):
            best_score = max_score
            best_match = cert_name
            best_word_count = matching_words
            if max_score == score1:
                match_type = "Fuzzy Ratio"
            elif max_score == score2:
                match_type = "Token Set Ratio"
            elif max_score == score3:
                match_type = "Token Sort Ratio"
            else:
                match_type = "Partial Ratio"
    
    if best_score >= threshold and best_word_count >= min_matching_words:
        return best_match, best_score, f"{match_type} ({best_word_count} kata cocok)"
    
    return None, best_score, "Tidak Ditemukan"

print("Fungsi matching berhasil dibuat!")
print("Filter: Minimal 2 kata yang cocok untuk dianggap match")

Fungsi matching berhasil dibuat!
Filter: Minimal 2 kata yang cocok untuk dianggap match


In [13]:
# ========== PISAHKAN PESERTA BERDASARKAN STATUS ==========
print(f"=" * 70)
print("PEMISAHAN DATA BERDASARKAN STATUS REGISTRASI")
print(f"=" * 70)

# Pisahkan peserta APPROVED dan NOT APPROVED
approved_df = peserta_df[peserta_df['Status'] == 'APPROVED'].copy()
not_approved_df = peserta_df[peserta_df['Status'] == 'NOT APPROVED'].copy()

print(f"Total peserta: {len(peserta_df)}")
print(f"  - Status APPROVED: {len(approved_df)} orang (tidak perlu di-crosscheck)")
print(f"  - Status NOT APPROVED: {len(not_approved_df)} orang (perlu di-crosscheck)")
print()

# ========== PROSES CROSS-CHECK HANYA UNTUK NOT APPROVED ==========
print(f"{'=' * 70}")
print("PROSES CROSS-CHECK UNTUK PESERTA NOT APPROVED")
print(f"=" * 70)

# Buat Full Name untuk certiport
certiport_mcf['Full Name'] = certiport_mcf['First Name'].fillna('') + ' ' + certiport_mcf['Last Name'].fillna('')
certiport_mos['Full Name'] = certiport_mos['First Name'].fillna('') + ' ' + certiport_mos['Last Name'].fillna('')

# Normalisasi nama certiport
certiport_mcf_names = [normalize_name(name) for name in certiport_mcf['Full Name'].tolist()]
certiport_mos_names = [normalize_name(name) for name in certiport_mos['Full Name'].tolist()]

# Hasil matching (HANYA UNTUK NOT APPROVED)
results = []

for idx, row in not_approved_df.iterrows():
    peserta_name = row['Nama']
    status_peserta = row['Status']
    sumber_data = row['Sumber']
    program = row['Program Dipilih']
    
    # Pilih data certiport sesuai sumber
    if sumber_data == 'MCF':
        certiport_names = certiport_mcf_names
    else:  # MOS
        certiport_names = certiport_mos_names
    
    match, score, match_type = find_best_match(peserta_name, certiport_names)
    
    results.append({
        'Nama Peserta': peserta_name,
        'NIM': row['NIM'],
        'Jurusan': row['Jurusan'],
        'Sumber Data': sumber_data,
        'Program': program,
        'Status Registrasi': status_peserta,
        'Nama di Certiport': match if match else '-',
        'Skor Kecocokan': score,
        'Tipe Match': match_type,
        'Status Certiport': '✅ DITEMUKAN' if match else '❌ TIDAK DITEMUKAN'
    })

# Buat DataFrame hasil (HANYA NOT APPROVED)
results_df = pd.DataFrame(results)
print(f"Proses cross-check selesai untuk {len(results)} peserta NOT APPROVED!")
print(f"\n--- Ringkasan Hasil Cross-Check (NOT APPROVED) ---")
print(results_df['Status Certiport'].value_counts())
print(f"\nBerdasarkan Sumber Data:")
print(results_df.groupby(['Sumber Data', 'Status Certiport']).size().unstack(fill_value=0))

PEMISAHAN DATA BERDASARKAN STATUS REGISTRASI
Total peserta: 157
  - Status APPROVED: 85 orang (tidak perlu di-crosscheck)
  - Status NOT APPROVED: 72 orang (perlu di-crosscheck)

PROSES CROSS-CHECK UNTUK PESERTA NOT APPROVED
Proses cross-check selesai untuk 72 peserta NOT APPROVED!

--- Ringkasan Hasil Cross-Check (NOT APPROVED) ---
Status Certiport
✅ DITEMUKAN          52
❌ TIDAK DITEMUKAN    20
Name: count, dtype: int64

Berdasarkan Sumber Data:
Status Certiport  ✅ DITEMUKAN  ❌ TIDAK DITEMUKAN
Sumber Data                                     
MCF                         3                 10
MOS                        49                 10


In [14]:
# ========== HASIL: PESERTA YANG DITEMUKAN DI CERTIPORT ==========
found_df = results_df[results_df['Status Certiport'] == '✅ DITEMUKAN'].copy()
print(f"=" * 70)
print(f"PESERTA YANG DITEMUKAN DI CERTIPORT: {len(found_df)} orang")
print(f"=" * 70)
print(found_df[['Nama Peserta', 'NIM', 'Sumber Data', 'Program', 'Status Registrasi', 'Nama di Certiport', 'Skor Kecocokan']].to_string(index=False))
print()

PESERTA YANG DITEMUKAN DI CERTIPORT: 52 orang
                      Nama Peserta       NIM Sumber Data           Program Status Registrasi                  Nama di Certiport  Skor Kecocokan
           MUHAMMAD RAFI AL ANBIYA 202231089         MCF MCF: Azure AI-900      NOT APPROVED               MUHAMMAD RAFI FARHAN              85
         MUHAMMAD RIFQI APRIANSYAH 202231104         MCF MCF: Azure AI-900      NOT APPROVED                     MUHAMMAD RIFQI             100
              UMMU PUTRI SALSABILA 202232036         MCF MCF: Azure AI-900      NOT APPROVED                SALSABILA EKA PUTRI              88
              RAIHAN CANDRA IRAWAN 202211117         MOS        MOS: Excel      NOT APPROVED                      RAIHAN IRAWAN             100
                   MUHAMMAD RAIHAN 202241017         MOS        MOS: Excel      NOT APPROVED                    MUHAMMAD RAIHAN             100
MUHAMMAD REVIANSYAH DANENDRA PUTRA 202131160         MOS         MOS: Word      NOT APPROV

In [15]:
# ========== HASIL: PESERTA YANG TIDAK DITEMUKAN DI CERTIPORT ==========
# Filter hanya yang status "NOT APPROVED" saja (sesuai data MCF/MOS)
not_found_df = results_df[
    (results_df['Status Certiport'] == '❌ TIDAK DITEMUKAN') & 
    (results_df['Status Registrasi'] == 'NOT APPROVED')
].copy()

print(f"=" * 70)
print(f"PESERTA NOT APPROVED YANG TIDAK DITEMUKAN DI CERTIPORT: {len(not_found_df)} orang")
print(f"=" * 70)
if len(not_found_df) > 0:
    print(not_found_df[['Nama Peserta', 'NIM', 'Sumber Data', 'Program', 'Status Registrasi', 'Skor Kecocokan']].to_string(index=False))
else:
    print("Semua peserta dengan status NOT APPROVED sudah ditemukan di Certiport!")
print()

# Tampilkan juga semua yang tidak ditemukan (semua status)
not_found_all_df = results_df[results_df['Status Certiport'] == '❌ TIDAK DITEMUKAN'].copy()
print(f"\n[INFO] Total peserta tidak ditemukan (semua status): {len(not_found_all_df)} orang")
print("Breakdown berdasarkan Status Registrasi:")
print(not_found_all_df['Status Registrasi'].value_counts().to_string())

PESERTA NOT APPROVED YANG TIDAK DITEMUKAN DI CERTIPORT: 20 orang
                        Nama Peserta       NIM Sumber Data           Program Status Registrasi  Skor Kecocokan
                  HABIL RAMA TAUFANY 202231077         MCF MCF: Azure AI-900      NOT APPROVED               0
                   LUTFIANSYAH MAJID 202131129         MCF MCF: Azure AI-900      NOT APPROVED               0
                KASHRINA MASYID AZKA 202231055         MCF MCF: Azure AI-900      NOT APPROVED               0
               MUHAMMAD NAUFAL ARIEF 202231026         MCF MCF: Azure AI-900      NOT APPROVED               0
      DESTA AZRA ROSANI FEERLIAN ADI 202232033         MCF MCF: Azure AI-900      NOT APPROVED               0
                    POPY LEONI PURBA 202231079         MCF MCF: Azure AI-900      NOT APPROVED               0
SERLINA MARGARET BANOET BR HUTAHAEAN 202131161         MCF MCF: Azure AI-900      NOT APPROVED               0
                        DEWI SAFITRI 202231011 

In [16]:
# ========== RINGKASAN STATISTIK ==========
print(f"=" * 70)
print("RINGKASAN LENGKAP")
print(f"=" * 70)
print(f"Total peserta dari registrasi   : {len(peserta_df)}")
print(f"  - Status APPROVED             : {len(approved_df)} orang")
print(f"  - Status NOT APPROVED         : {len(not_approved_df)} orang")
print(f"\nTotal data di Certiport         : {len(certiport_df)}")
print(f"  - MCF (AI-900)                : {len(certiport_mcf)}")
print(f"  - MOS (Office 2019)           : {len(certiport_mos)}")
print(f"-" * 70)
print("HASIL CROSS-CHECK (HANYA NOT APPROVED):")
print(f"✅ Ditemukan di Certiport       : {len(found_df)} ({len(found_df)/len(not_approved_df)*100:.1f}% dari NOT APPROVED)")
print(f"❌ Tidak ditemukan di Certiport : {len(not_found_all_df)} ({len(not_found_all_df)/len(not_approved_df)*100:.1f}% dari NOT APPROVED)")
print(f"=" * 70)

# Breakdown berdasarkan tipe match
print("\nBreakdown berdasarkan tipe kecocokan:")
print(results_df['Tipe Match'].value_counts().to_string())

# Breakdown berdasarkan sumber data
print("\nBreakdown hasil cross-check berdasarkan Sumber Data:")
pivot = results_df.groupby(['Sumber Data', 'Status Certiport']).size().unstack(fill_value=0)
print(pivot.to_string())

RINGKASAN LENGKAP
Total peserta dari registrasi   : 157
  - Status APPROVED             : 85 orang
  - Status NOT APPROVED         : 72 orang

Total data di Certiport         : 4853
  - MCF (AI-900)                : 897
  - MOS (Office 2019)           : 3445
----------------------------------------------------------------------
HASIL CROSS-CHECK (HANYA NOT APPROVED):
✅ Ditemukan di Certiport       : 52 (72.2% dari NOT APPROVED)
❌ Tidak ditemukan di Certiport : 20 (27.8% dari NOT APPROVED)

Breakdown berdasarkan tipe kecocokan:
Tipe Match
Token Set Ratio (2 kata cocok)    37
Tidak Ditemukan                   20
Exact Match                       11
Token Set Ratio (3 kata cocok)     2
Partial Ratio (2 kata cocok)       1
Token Set Ratio (4 kata cocok)     1

Breakdown hasil cross-check berdasarkan Sumber Data:
Status Certiport  ✅ DITEMUKAN  ❌ TIDAK DITEMUKAN
Sumber Data                                     
MCF                         3                 10
MOS                        49    

In [17]:
# ========== EXPORT HASIL KE CSV ==========
print(f"\n{'=' * 70}")
print("EXPORT HASIL KE FILE CSV")
print(f"=" * 70)

# 1. Export peserta APPROVED (langsung diterima, tidak perlu crosscheck)
approved_df.to_csv('peserta_approved.csv', index=False, encoding='utf-8-sig')

# 2. Export semua hasil cross-check (NOT APPROVED)
results_df.to_csv('hasil_crosscheck_not_approved.csv', index=False, encoding='utf-8-sig')

# 3. Export NOT APPROVED yang ditemukan di Certiport
found_df.to_csv('peserta_not_approved_ditemukan.csv', index=False, encoding='utf-8-sig')

# 4. Export NOT APPROVED yang tidak ditemukan di Certiport
not_found_all_df.to_csv('peserta_not_approved_tidak_ditemukan.csv', index=False, encoding='utf-8-sig')

print("✅ File hasil telah disimpan:")
print(f"\n📁 PESERTA APPROVED (Langsung Diterima):")
print(f"   1. peserta_approved.csv")
print(f"      → {len(approved_df)} peserta dengan status APPROVED")
print(f"\n📁 HASIL CROSS-CHECK NOT APPROVED:")
print(f"   2. hasil_crosscheck_not_approved.csv")
print(f"      → Semua hasil cross-check peserta NOT APPROVED ({len(results_df)} peserta)")
print(f"   3. peserta_not_approved_ditemukan.csv")
print(f"      → NOT APPROVED yang ditemukan di Certiport ({len(found_df)} peserta)")
print(f"   4. peserta_not_approved_tidak_ditemukan.csv")
print(f"      → NOT APPROVED yang TIDAK ditemukan di Certiport ({len(not_found_all_df)} peserta)")
print(f"\n{'=' * 70}")


EXPORT HASIL KE FILE CSV
✅ File hasil telah disimpan:

📁 PESERTA APPROVED (Langsung Diterima):
   1. peserta_approved.csv
      → 85 peserta dengan status APPROVED

📁 HASIL CROSS-CHECK NOT APPROVED:
   2. hasil_crosscheck_not_approved.csv
      → Semua hasil cross-check peserta NOT APPROVED (72 peserta)
   3. peserta_not_approved_ditemukan.csv
      → NOT APPROVED yang ditemukan di Certiport (52 peserta)
   4. peserta_not_approved_tidak_ditemukan.csv
      → NOT APPROVED yang TIDAK ditemukan di Certiport (20 peserta)



In [18]:
# ========== FITUR 1: REKOMENDASI APPROVAL OTOMATIS ==========
print(f"\n{'=' * 70}")
print("⭐ REKOMENDASI PESERTA UNTUK DI-APPROVE")
print(f"=" * 70)

# Peserta yang LAYAK di-approve: NOT APPROVED + TIDAK DITEMUKAN di Certiport (belum pernah ujian)
recommended_for_approval = not_found_all_df.copy()
recommended_for_approval['Rekomendasi'] = '✅ APPROVE - Belum Pernah Ujian'
recommended_for_approval['Alasan'] = 'Peserta belum pernah mengikuti ujian di Certiport'

# Peserta yang TIDAK layak di-approve: NOT APPROVED + SUDAH DITEMUKAN di Certiport (sudah ujian)
not_recommended = found_df.copy()
not_recommended['Rekomendasi'] = '❌ JANGAN APPROVE - Sudah Pernah Ujian'
not_recommended['Alasan'] = 'Peserta sudah pernah ujian (nama ditemukan di database Certiport)'

print(f"✅ REKOMENDASI APPROVE: {len(recommended_for_approval)} peserta")
print(f"   → Peserta ini belum pernah ujian, layak untuk disetujui")
print(f"\n❌ TIDAK DIREKOMENDASIKAN: {len(not_recommended)} peserta")
print(f"   → Peserta ini sudah pernah ujian (ditemukan di Certiport)")

# Breakdown per program
print(f"\n--- Breakdown Rekomendasi per Sumber ---")
print("\n[DIREKOMENDASIKAN untuk APPROVE]:")
if len(recommended_for_approval) > 0:
    print(recommended_for_approval.groupby(['Sumber Data']).size().to_string())
else:
    print("Tidak ada")

print("\n[TIDAK DIREKOMENDASIKAN - sudah ujian]:")
if len(not_recommended) > 0:
    print(not_recommended.groupby(['Sumber Data']).size().to_string())
else:
    print("Tidak ada")

# Tampilkan detail
print(f"\n{'=' * 70}")
print("DETAIL PESERTA YANG DIREKOMENDASIKAN UNTUK APPROVE")
print(f"=" * 70)
if len(recommended_for_approval) > 0:
    print(recommended_for_approval[['Nama Peserta', 'NIM', 'Sumber Data', 'Program', 'Rekomendasi']].to_string(index=False))
else:
    print("Tidak ada peserta yang direkomendasikan untuk di-approve")

# Export rekomendasi
recommended_for_approval.to_csv('APPROVE_INI_belum_pernah_ujian.csv', index=False, encoding='utf-8-sig')
not_recommended.to_csv('JANGAN_APPROVE_sudah_ujian.csv', index=False, encoding='utf-8-sig')

print(f"\n{'=' * 70}")
print("✅ File rekomendasi telah disimpan:")
print(f"   📄 APPROVE_INI_belum_pernah_ujian.csv")
print(f"      → {len(recommended_for_approval)} peserta yang LAYAK DI-APPROVE")
print(f"   📄 JANGAN_APPROVE_sudah_ujian.csv")
print(f"      → {len(not_recommended)} peserta yang JANGAN DI-APPROVE (sudah ujian)")
print(f"=" * 70)


⭐ REKOMENDASI PESERTA UNTUK DI-APPROVE
✅ REKOMENDASI APPROVE: 20 peserta
   → Peserta ini belum pernah ujian, layak untuk disetujui

❌ TIDAK DIREKOMENDASIKAN: 52 peserta
   → Peserta ini sudah pernah ujian (ditemukan di Certiport)

--- Breakdown Rekomendasi per Sumber ---

[DIREKOMENDASIKAN untuk APPROVE]:
Sumber Data
MCF    10
MOS    10

[TIDAK DIREKOMENDASIKAN - sudah ujian]:
Sumber Data
MCF     3
MOS    49

DETAIL PESERTA YANG DIREKOMENDASIKAN UNTUK APPROVE
                        Nama Peserta       NIM Sumber Data           Program                    Rekomendasi
                  HABIL RAMA TAUFANY 202231077         MCF MCF: Azure AI-900 ✅ APPROVE - Belum Pernah Ujian
                   LUTFIANSYAH MAJID 202131129         MCF MCF: Azure AI-900 ✅ APPROVE - Belum Pernah Ujian
                KASHRINA MASYID AZKA 202231055         MCF MCF: Azure AI-900 ✅ APPROVE - Belum Pernah Ujian
               MUHAMMAD NAUFAL ARIEF 202231026         MCF MCF: Azure AI-900 ✅ APPROVE - Belum Pernah 

In [19]:
# ========== FITUR 2: DETEKSI PESERTA YANG PERNAH GAGAL ==========
print(f"\n{'=' * 70}")
print("⚠️  DETEKSI PESERTA YANG PERNAH GAGAL UJIAN")
print(f"=" * 70)

# Gabungkan data MCF dan MOS Certiport
all_certiport = pd.concat([certiport_mcf, certiport_mos], ignore_index=True)

# Filter peserta yang pernah gagal
if 'Result' in all_certiport.columns:
    failed_students = all_certiport[all_certiport['Result'] == 'Fail'].copy()
    
    print(f"Total peserta yang pernah GAGAL di Certiport: {len(failed_students)}")
    
    if len(failed_students) > 0:
        # Buat Full Name Normalized untuk failed students
        failed_students['Full Name Normalized'] = failed_students.apply(
            lambda x: normalize_name(f"{x['First Name']} {x['Last Name']}"), axis=1
        )
        
        # Tambahkan kolom Nama Normalized ke not_approved_df
        not_approved_df_temp = not_approved_df.copy()
        not_approved_df_temp['Nama Normalized'] = not_approved_df_temp['Nama'].apply(normalize_name)
        
        # Cari yang pernah gagal dan daftar lagi
        daftar_lagi = []
        for idx, row in not_approved_df_temp.iterrows():
            nama_normalized = row['Nama Normalized']
            sumber = row['Sumber']
            
            # Filter berdasarkan sumber (MCF cek MCF, MOS cek MOS)
            if sumber == 'MCF':
                failed_to_check = failed_students[failed_students['Exam'].str.contains('AI-900', na=False)]
            else:
                failed_to_check = failed_students[failed_students['Exam'].str.contains('Office 2019', na=False)]
            
            # Cek kecocokan dengan fuzzy matching
            for _, failed_row in failed_to_check.iterrows():
                failed_name = failed_row['Full Name Normalized']
                # Cek exact match atau fuzzy match tinggi
                score = fuzz.token_set_ratio(nama_normalized, failed_name)
                if score >= 85 or nama_normalized == failed_name:
                    daftar_lagi.append({
                        'Nama Peserta': row['Nama'],
                        'NIM': row['NIM'],
                        'Sumber Data': sumber,
                        'Program Daftar': row['Program Dipilih'],
                        'Status': '⚠️ PERNAH GAGAL - DAFTAR LAGI',
                        'Nama di Certiport': f"{failed_row['First Name']} {failed_row['Last Name']}",
                        'Exam Sebelumnya': failed_row['Exam'],
                        'Tanggal Ujian': failed_row['Exam Date'] if 'Exam Date' in failed_row else '-',
                        'Skor Sebelumnya': failed_row['Score'] if 'Score' in failed_row else '-',
                        'Skor Kecocokan Nama': score,
                        'Keputusan': '❓ PERLU REVIEW MANUAL'
                    })
                    break  # Hanya ambil 1 match per peserta
        
        if len(daftar_lagi) > 0:
            daftar_lagi_df = pd.DataFrame(daftar_lagi)
            print(f"\n⚠️  PERHATIAN: {len(daftar_lagi_df)} peserta PERNAH GAGAL dan DAFTAR LAGI!")
            print(f"\n--- Detail Peserta yang Pernah Gagal ---")
            print(daftar_lagi_df[['Nama Peserta', 'NIM', 'Sumber Data', 'Exam Sebelumnya', 'Skor Sebelumnya', 'Keputusan']].to_string(index=False))
            
            daftar_lagi_df.to_csv('PERHATIAN_peserta_gagal_daftar_lagi.csv', index=False, encoding='utf-8-sig')
            print(f"\n📄 File PERHATIAN_peserta_gagal_daftar_lagi.csv telah disimpan")
            print(f"\n💡 SARAN: Review manual diperlukan untuk peserta ini.")
            print(f"   Pertimbangkan: apakah boleh mengikuti ujian lagi?")
        else:
            print("\n✅ Tidak ada peserta NOT APPROVED yang pernah gagal ujian sebelumnya")
            daftar_lagi_df = pd.DataFrame()  # Empty dataframe
    else:
        print("\n✅ Tidak ada data peserta yang gagal di Certiport")
        daftar_lagi_df = pd.DataFrame()
else:
    print("\n⚠️  Kolom 'Result' tidak ditemukan di data Certiport")
    daftar_lagi_df = pd.DataFrame()


⚠️  DETEKSI PESERTA YANG PERNAH GAGAL UJIAN
Total peserta yang pernah GAGAL di Certiport: 1714

⚠️  PERHATIAN: 44 peserta PERNAH GAGAL dan DAFTAR LAGI!

--- Detail Peserta yang Pernah Gagal ---
                      Nama Peserta       NIM Sumber Data                    Exam Sebelumnya  Skor Sebelumnya             Keputusan
              RAIHAN CANDRA IRAWAN 202211117         MOS       Microsoft Word (Office 2019)            668.0 ❓ PERLU REVIEW MANUAL
                   MUHAMMAD RAIHAN 202241017         MOS       Microsoft Word (Office 2019)            636.0 ❓ PERLU REVIEW MANUAL
MUHAMMAD REVIANSYAH DANENDRA PUTRA 202131160         MOS       Microsoft Word (Office 2019)            152.0 ❓ PERLU REVIEW MANUAL
   DHEA HERAWATI INDAH PUTRI ERWIN 202231088         MOS       Microsoft Word (Office 2019)            127.0 ❓ PERLU REVIEW MANUAL
               BINTAR ANDIKA PUTRA 202211113         MOS Microsoft PowerPoint (Office 2019)            547.0 ❓ PERLU REVIEW MANUAL
          RACHMAD F

In [20]:
# ========== FITUR 3: DASHBOARD SUMMARY ==========
print(f"\n{'=' * 100}")
print(" " * 35 + "📊 DASHBOARD SUMMARY")
print(f"=" * 100)

total_peserta = len(peserta_df)
total_not_approved = len(not_approved_df)

# Buat tabel summary
print(f"""
┌─────────────────────────────────────────────────────────────────────────────────┐
│                              DATA PENDAFTAR                                      │
├─────────────────────────────────────────────────────────────────────────────────┤
│  Total Pendaftar              : {total_peserta:>5} peserta                                  │
│    ├─ MCF (Azure AI-900)      : {len(peserta_mcf_df):>5} peserta ({len(peserta_mcf_df)/total_peserta*100:>5.1f}%)                       │
│    └─ MOS (Office 2019)       : {len(peserta_mos_df):>5} peserta ({len(peserta_mos_df)/total_peserta*100:>5.1f}%)                       │
├─────────────────────────────────────────────────────────────────────────────────┤
│                            STATUS REGISTRASI                                     │
├─────────────────────────────────────────────────────────────────────────────────┤
│  ✅ Status APPROVED           : {len(approved_df):>5} peserta ({len(approved_df)/total_peserta*100:>5.1f}%)                       │
│  ⏳ Status NOT APPROVED       : {len(not_approved_df):>5} peserta ({len(not_approved_df)/total_peserta*100:>5.1f}%)                       │
├─────────────────────────────────────────────────────────────────────────────────┤
│                           DATA CERTIPORT                                         │
├─────────────────────────────────────────────────────────────────────────────────┤
│  Total Data Certiport         : {len(certiport_df):>5} records                                  │
│    ├─ MCF (AI-900)            : {len(certiport_mcf):>5} records                                  │
│    └─ MOS (Office 2019)       : {len(certiport_mos):>5} records                                  │
├─────────────────────────────────────────────────────────────────────────────────┤
│                      HASIL CROSS-CHECK (NOT APPROVED)                            │
├─────────────────────────────────────────────────────────────────────────────────┤
│  ✅ Ditemukan di Certiport    : {len(found_df):>5} peserta ({len(found_df)/total_not_approved*100 if total_not_approved > 0 else 0:>5.1f}%)  → JANGAN APPROVE    │
│  ❌ Tidak ditemukan           : {len(not_found_all_df):>5} peserta ({len(not_found_all_df)/total_not_approved*100 if total_not_approved > 0 else 0:>5.1f}%)  → APPROVE INI       │
├─────────────────────────────────────────────────────────────────────────────────┤
│  ⚠️  Pernah Gagal Daftar Lagi : {len(daftar_lagi_df) if 'daftar_lagi_df' in dir() else 0:>5} peserta          → PERLU REVIEW      │
└─────────────────────────────────────────────────────────────────────────────────┘
""")

print(f"\n{'=' * 100}")
print("🎯 REKOMENDASI AKHIR:")
print(f"   ✅ APPROVE {len(recommended_for_approval)} peserta (belum pernah ujian)")
print(f"   ❌ JANGAN APPROVE {len(not_recommended)} peserta (sudah pernah ujian)")
if 'daftar_lagi_df' in dir() and len(daftar_lagi_df) > 0:
    print(f"   ⚠️  REVIEW MANUAL {len(daftar_lagi_df)} peserta (pernah gagal, daftar lagi)")
print(f"=" * 100)


                                   📊 DASHBOARD SUMMARY

┌─────────────────────────────────────────────────────────────────────────────────┐
│                              DATA PENDAFTAR                                      │
├─────────────────────────────────────────────────────────────────────────────────┤
│  Total Pendaftar              :   157 peserta                                  │
│    ├─ MCF (Azure AI-900)      :    56 peserta ( 35.7%)                       │
│    └─ MOS (Office 2019)       :   101 peserta ( 64.3%)                       │
├─────────────────────────────────────────────────────────────────────────────────┤
│                            STATUS REGISTRASI                                     │
├─────────────────────────────────────────────────────────────────────────────────┤
│  ✅ Status APPROVED           :    85 peserta ( 54.1%)                       │
│  ⏳ Status NOT APPROVED       :    72 peserta ( 45.9%)                       │
├───────────────────────────────

In [21]:
# ========== FITUR 4: FILTER LANJUTAN BERDASARKAN CONFIDENCE ==========
print(f"\n{'=' * 70}")
print("🔍 FILTER LANJUTAN - ANALISIS TINGKAT KEPERCAYAAN MATCHING")
print(f"=" * 70)

# Kategorisasi berdasarkan skor kecocokan
def kategorisasi_confidence(row):
    score = row['Skor Kecocokan']
    tipe = row['Tipe Match']
    
    if tipe == 'Exact Match' or tipe == 'Exact Match (Nama Terbalik)':
        return 'HIGH - Exact Match'
    elif score >= 95:
        return 'HIGH - Sangat Yakin'
    elif score >= 85:
        return 'MEDIUM - Cukup Yakin'
    elif score >= 80:
        return 'LOW - Perlu Verifikasi'
    else:
        return 'NOT MATCHED'

# Terapkan kategorisasi ke data yang ditemukan
if len(found_df) > 0:
    found_df['Confidence Level'] = found_df.apply(kategorisasi_confidence, axis=1)
    
    print("\n📊 BREAKDOWN TINGKAT KEPERCAYAAN MATCHING:")
    print(found_df['Confidence Level'].value_counts().to_string())
    
    # Filter berdasarkan confidence
    high_confidence = found_df[found_df['Confidence Level'].str.contains('HIGH')]
    medium_confidence = found_df[found_df['Confidence Level'] == 'MEDIUM - Cukup Yakin']
    low_confidence = found_df[found_df['Confidence Level'] == 'LOW - Perlu Verifikasi']
    
    print(f"\n--- HIGH CONFIDENCE ({len(high_confidence)} peserta) ---")
    print("→ Nama cocok dengan sangat akurat, YAKIN sudah pernah ujian")
    if len(high_confidence) > 0:
        print(high_confidence[['Nama Peserta', 'Nama di Certiport', 'Skor Kecocokan', 'Confidence Level']].to_string(index=False))
    
    print(f"\n--- MEDIUM CONFIDENCE ({len(medium_confidence)} peserta) ---")
    print("→ Nama cukup cocok, kemungkinan besar sudah pernah ujian")
    if len(medium_confidence) > 0:
        print(medium_confidence[['Nama Peserta', 'Nama di Certiport', 'Skor Kecocokan', 'Confidence Level']].to_string(index=False))
    
    print(f"\n--- LOW CONFIDENCE ({len(low_confidence)} peserta) ---")
    print("→ ⚠️ PERLU VERIFIKASI MANUAL - Nama mirip tapi tidak yakin 100%")
    if len(low_confidence) > 0:
        print(low_confidence[['Nama Peserta', 'Nama di Certiport', 'Skor Kecocokan', 'Confidence Level']].to_string(index=False))
    
    # Deteksi kemungkinan false positive (nama yang mungkin salah match)
    print(f"\n{'=' * 70}")
    print("⚠️  DETEKSI KEMUNGKINAN FALSE POSITIVE (SALAH MATCHING)")
    print(f"=" * 70)
    
    suspicious_matches = found_df[
        (found_df['Skor Kecocokan'] < 90) | 
        (found_df['Confidence Level'] == 'LOW - Perlu Verifikasi')
    ].copy()
    
    if len(suspicious_matches) > 0:
        print(f"Ditemukan {len(suspicious_matches)} peserta yang PERLU DICEK ULANG:")
        print("(Kemungkinan nama yang cocok adalah orang berbeda)")
        print()
        for idx, row in suspicious_matches.iterrows():
            print(f"  ❓ {row['Nama Peserta']}")
            print(f"     → Cocok dengan: {row['Nama di Certiport']}")
            print(f"     → Skor: {row['Skor Kecocokan']} | {row['Confidence Level']}")
            print()
        
        suspicious_matches.to_csv('VERIFIKASI_matching_meragukan.csv', index=False, encoding='utf-8-sig')
        print(f"📄 File VERIFIKASI_matching_meragukan.csv telah disimpan")
    else:
        print("✅ Semua matching memiliki tingkat kepercayaan tinggi")
else:
    print("✅ Tidak ada peserta yang ditemukan di Certiport (semua baru)")

# Update found_df untuk export
if len(found_df) > 0:
    found_df.to_csv('JANGAN_APPROVE_sudah_ujian.csv', index=False, encoding='utf-8-sig')


🔍 FILTER LANJUTAN - ANALISIS TINGKAT KEPERCAYAAN MATCHING

📊 BREAKDOWN TINGKAT KEPERCAYAAN MATCHING:
Confidence Level
HIGH - Sangat Yakin       37
HIGH - Exact Match        11
MEDIUM - Cukup Yakin       2
LOW - Perlu Verifikasi     2

--- HIGH CONFIDENCE (48 peserta) ---
→ Nama cocok dengan sangat akurat, YAKIN sudah pernah ujian
                      Nama Peserta                  Nama di Certiport  Skor Kecocokan    Confidence Level
         MUHAMMAD RIFQI APRIANSYAH                     MUHAMMAD RIFQI             100 HIGH - Sangat Yakin
              RAIHAN CANDRA IRAWAN                      RAIHAN IRAWAN             100 HIGH - Sangat Yakin
                   MUHAMMAD RAIHAN                    MUHAMMAD RAIHAN             100  HIGH - Exact Match
MUHAMMAD REVIANSYAH DANENDRA PUTRA MUHAMMAD REVIANSYAH DANENDRA PUTRA             100  HIGH - Exact Match
   DHEA HERAWATI INDAH PUTRI ERWIN          DHEA HERAWATI PUTRI ERWIN             100 HIGH - Sangat Yakin
         ANDI MUHAMMAD NABIL AL

In [22]:
# ========== FITUR 5: EXPORT KE EXCEL DENGAN MULTIPLE SHEETS ==========
print(f"\n{'=' * 70}")
print("📊 EXPORT KE EXCEL DENGAN MULTIPLE SHEETS & HIGHLIGHTING")
print(f"=" * 70)

# Install openpyxl jika belum ada
try:
    from openpyxl import Workbook
    from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
    from openpyxl.utils.dataframe import dataframe_to_rows
    print("✅ Library openpyxl tersedia")
except ImportError:
    print("⚠️ Installing openpyxl...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'openpyxl'])
    from openpyxl import Workbook
    from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
    from openpyxl.utils.dataframe import dataframe_to_rows
    print("✅ Library openpyxl berhasil diinstall")

# Buat workbook baru
wb = Workbook()

# Style definitions
header_fill = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True)
approve_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")  # Hijau muda
reject_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")    # Merah muda
warning_fill = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")   # Kuning muda
thin_border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)

def style_worksheet(ws, df, highlight_col=None, approve_value=None, reject_value=None):
    """Apply styling to worksheet"""
    # Header styling
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center', vertical='center')
        cell.border = thin_border
    
    # Data styling
    for row_idx, row in enumerate(ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=ws.max_column), start=2):
        for cell in row:
            cell.border = thin_border
            cell.alignment = Alignment(vertical='center')
        
        # Highlight berdasarkan kolom tertentu
        if highlight_col and highlight_col <= len(row):
            cell_value = str(ws.cell(row=row_idx, column=highlight_col).value)
            if approve_value and approve_value in cell_value:
                for cell in row:
                    cell.fill = approve_fill
            elif reject_value and reject_value in cell_value:
                for cell in row:
                    cell.fill = reject_fill
            elif '⚠️' in cell_value or 'REVIEW' in cell_value or 'LOW' in cell_value:
                for cell in row:
                    cell.fill = warning_fill
    
    # Auto-adjust column width
    for column in ws.columns:
        max_length = 0
        column_letter = column[0].column_letter
        for cell in column:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        adjusted_width = min(max_length + 2, 50)
        ws.column_dimensions[column_letter].width = adjusted_width

# Sheet 1: Dashboard Summary
ws1 = wb.active
ws1.title = "Dashboard Summary"
summary_data = [
    ["KATEGORI", "JUMLAH", "PERSENTASE", "STATUS"],
    ["Total Pendaftar", len(peserta_df), "100%", "ℹ️"],
    ["  ├─ MCF (Azure AI-900)", len(peserta_mcf_df), f"{len(peserta_mcf_df)/len(peserta_df)*100:.1f}%", ""],
    ["  └─ MOS (Office 2019)", len(peserta_mos_df), f"{len(peserta_mos_df)/len(peserta_df)*100:.1f}%", ""],
    ["", "", "", ""],
    ["Status APPROVED", len(approved_df), f"{len(approved_df)/len(peserta_df)*100:.1f}%", "✅"],
    ["Status NOT APPROVED", len(not_approved_df), f"{len(not_approved_df)/len(peserta_df)*100:.1f}%", "⏳"],
    ["", "", "", ""],
    ["NOT APPROVED - Sudah Ujian", len(found_df), f"{len(found_df)/len(not_approved_df)*100:.1f}%" if len(not_approved_df) > 0 else "0%", "❌ JANGAN APPROVE"],
    ["NOT APPROVED - Belum Ujian", len(not_found_all_df), f"{len(not_found_all_df)/len(not_approved_df)*100:.1f}%" if len(not_approved_df) > 0 else "0%", "✅ APPROVE INI"],
]
for row in summary_data:
    ws1.append(row)
style_worksheet(ws1, None, highlight_col=4, approve_value="APPROVE INI", reject_value="JANGAN")

# Sheet 2: Rekomendasi APPROVE
ws2 = wb.create_sheet("APPROVE - Belum Ujian")
if len(recommended_for_approval) > 0:
    for r_idx, row in enumerate(dataframe_to_rows(recommended_for_approval[['Nama Peserta', 'NIM', 'Jurusan', 'Sumber Data', 'Program', 'Rekomendasi']], index=False, header=True), 1):
        ws2.append(row)
    style_worksheet(ws2, recommended_for_approval, highlight_col=6, approve_value="APPROVE")
else:
    ws2.append(["Tidak ada peserta yang direkomendasikan untuk approve"])

# Sheet 3: JANGAN APPROVE
ws3 = wb.create_sheet("JANGAN APPROVE - Sudah Ujian")
if len(not_recommended) > 0:
    cols_to_export = ['Nama Peserta', 'NIM', 'Sumber Data', 'Program', 'Nama di Certiport', 'Skor Kecocokan', 'Rekomendasi']
    if 'Confidence Level' in found_df.columns:
        cols_to_export.append('Confidence Level')
    export_df = not_recommended[[c for c in cols_to_export if c in not_recommended.columns]]
    for r_idx, row in enumerate(dataframe_to_rows(export_df, index=False, header=True), 1):
        ws3.append(row)
    style_worksheet(ws3, export_df, highlight_col=7, reject_value="JANGAN")
else:
    ws3.append(["Tidak ada peserta yang sudah pernah ujian"])

# Sheet 4: Perlu Verifikasi (Low Confidence)
ws4 = wb.create_sheet("VERIFIKASI - Low Confidence")
if len(found_df) > 0 and 'Confidence Level' in found_df.columns:
    low_conf = found_df[found_df['Confidence Level'] == 'LOW - Perlu Verifikasi']
    if len(low_conf) > 0:
        for r_idx, row in enumerate(dataframe_to_rows(low_conf[['Nama Peserta', 'NIM', 'Nama di Certiport', 'Skor Kecocokan', 'Confidence Level']], index=False, header=True), 1):
            ws4.append(row)
        style_worksheet(ws4, low_conf)
    else:
        ws4.append(["Tidak ada matching dengan confidence rendah"])
else:
    ws4.append(["Tidak ada data yang perlu diverifikasi"])

# Sheet 5: Peserta Gagal Daftar Lagi
ws5 = wb.create_sheet("REVIEW - Pernah Gagal")
if 'daftar_lagi_df' in dir() and len(daftar_lagi_df) > 0:
    for r_idx, row in enumerate(dataframe_to_rows(daftar_lagi_df, index=False, header=True), 1):
        ws5.append(row)
    style_worksheet(ws5, daftar_lagi_df)
else:
    ws5.append(["Tidak ada peserta yang pernah gagal dan daftar lagi"])

# Sheet 6: Semua Hasil Cross-Check
ws6 = wb.create_sheet("Semua Hasil Cross-Check")
for r_idx, row in enumerate(dataframe_to_rows(results_df, index=False, header=True), 1):
    ws6.append(row)
style_worksheet(ws6, results_df, highlight_col=10, approve_value="TIDAK DITEMUKAN", reject_value="DITEMUKAN")

# Simpan file Excel
excel_filename = 'HASIL_FILTER_PESERTA_LENGKAP.xlsx'
wb.save(excel_filename)

print(f"\n✅ File Excel berhasil disimpan: {excel_filename}")
print(f"\n📋 DAFTAR SHEETS DALAM FILE EXCEL:")
print(f"   1. Dashboard Summary      - Ringkasan keseluruhan")
print(f"   2. APPROVE - Belum Ujian  - 🟢 Peserta yang layak di-approve ({len(recommended_for_approval)} peserta)")
print(f"   3. JANGAN APPROVE         - 🔴 Peserta yang sudah ujian ({len(not_recommended)} peserta)")
print(f"   4. VERIFIKASI             - 🟡 Matching perlu dicek ulang")
print(f"   5. REVIEW - Pernah Gagal  - 🟡 Peserta pernah gagal daftar lagi")
print(f"   6. Semua Hasil Cross-Check - Detail lengkap semua peserta")
print(f"\n💡 Tips: Buka file Excel, gunakan filter & sorting untuk review lebih mudah!")
print(f"   - Hijau = Layak di-approve")
print(f"   - Merah = Jangan di-approve")
print(f"   - Kuning = Perlu review manual")


📊 EXPORT KE EXCEL DENGAN MULTIPLE SHEETS & HIGHLIGHTING
✅ Library openpyxl tersedia

✅ File Excel berhasil disimpan: HASIL_FILTER_PESERTA_LENGKAP.xlsx

📋 DAFTAR SHEETS DALAM FILE EXCEL:
   1. Dashboard Summary      - Ringkasan keseluruhan
   2. APPROVE - Belum Ujian  - 🟢 Peserta yang layak di-approve (20 peserta)
   3. JANGAN APPROVE         - 🔴 Peserta yang sudah ujian (52 peserta)
   4. VERIFIKASI             - 🟡 Matching perlu dicek ulang
   5. REVIEW - Pernah Gagal  - 🟡 Peserta pernah gagal daftar lagi
   6. Semua Hasil Cross-Check - Detail lengkap semua peserta

💡 Tips: Buka file Excel, gunakan filter & sorting untuk review lebih mudah!
   - Hijau = Layak di-approve
   - Merah = Jangan di-approve
   - Kuning = Perlu review manual


In [24]:
# ========== FITUR PENCARIAN NAMA MANUAL ==========
print(f"\n{'=' * 70}")
print("🔎 FITUR PENCARIAN NAMA")
print(f"=" * 70)

def cari_nama(nama_peserta, sumber='ALL', threshold=70):
    """
    Mencari nama peserta di database Certiport
    
    Parameters:
    - nama_peserta: Nama yang ingin dicari
    - sumber: 'MCF', 'MOS', atau 'ALL' (default)
    - threshold: Batas minimum skor kecocokan (default 70)
    """
    print(f"\n{'=' * 70}")
    print(f"🔍 Mencari: {nama_peserta}")
    print(f"   Sumber: {sumber}")
    print(f"=" * 70)
    
    # Tentukan database yang akan dicari
    search_databases = []
    if sumber == 'MCF' or sumber == 'ALL':
        search_databases.append(('MCF (Azure AI-900)', certiport_mcf_names, certiport_mcf))
    if sumber == 'MOS' or sumber == 'ALL':
        search_databases.append(('MOS (Office 2019)', certiport_mos_names, certiport_mos))
    
    found_any = False
    all_similar = []
    
    for db_name, name_list, df_original in search_databases:
        print(f"\n--- Mencari di {db_name} ---")
        
        match, score, match_type = find_best_match(nama_peserta, name_list, threshold)
        
        if match:
            found_any = True
            print(f"✅ DITEMUKAN!")
            print(f"   Nama di Certiport: {match}")
            print(f"   Skor kecocokan: {score}")
            print(f"   Tipe match: {match_type}")
            
            # Cari detail di dataframe asli
            matched_rows = df_original[
                (df_original['First Name'].fillna('') + ' ' + df_original['Last Name'].fillna('')).apply(
                    lambda x: normalize_name(x) == match
                )
            ]
            
            if len(matched_rows) > 0:
                row = matched_rows.iloc[0]
                print(f"\n   📋 Detail dari Certiport:")
                print(f"      Exam: {row['Exam'] if 'Exam' in row else '-'}")
                print(f"      Result: {row['Result'] if 'Result' in row else '-'}")
                print(f"      Score: {row['Score'] if 'Score' in row else '-'}")
                print(f"      Exam Date: {row['Exam Date'] if 'Exam Date' in row else '-'}")
        else:
            print(f"❌ Tidak ditemukan dengan threshold {threshold}")
            print(f"   Skor tertinggi: {score}")
        
        # Kumpulkan nama-nama yang mirip
        for cert_name in name_list:
            scores = [
                fuzz.ratio(normalize_name(nama_peserta), cert_name),
                fuzz.token_set_ratio(normalize_name(nama_peserta), cert_name),
                fuzz.token_sort_ratio(normalize_name(nama_peserta), cert_name)
            ]
            max_score = max(scores)
            if max_score >= threshold - 20:
                all_similar.append((cert_name, max_score, db_name))
    
    # Tampilkan nama-nama yang mirip
    if not found_any and len(all_similar) > 0:
        print(f"\n{'=' * 70}")
        print(f"🔍 Nama-nama yang mirip (skor >= {threshold - 20}):")
        print(f"=" * 70)
        # Sort by score descending
        all_similar.sort(key=lambda x: x[1], reverse=True)
        for cert_name, sim_score, db_name in all_similar[:5]:  # Top 5
            print(f"   {sim_score:>3} | {cert_name} ({db_name})")
    
    if not found_any and len(all_similar) == 0:
        print(f"\n❌ Tidak ada nama yang mirip ditemukan")
        print(f"💡 Tips: Coba turunkan threshold atau periksa ejaan nama")
    
    print(f"\n{'=' * 70}")

print("✅ Fungsi cari_nama() siap digunakan!")
print("\n📖 Cara menggunakan:")
print("   1. cari_nama('NAMA LENGKAP')               → Cari di MCF & MOS")
print("   2. cari_nama('NAMA LENGKAP', 'MCF')        → Cari di MCF saja")
print("   3. cari_nama('NAMA LENGKAP', 'MOS')        → Cari di MOS saja")
print("   4. cari_nama('NAMA LENGKAP', threshold=60) → Threshold lebih rendah")
print("\n🔍 Contoh:")
print("   cari_nama('LILY FITRI HASANAH')")
print("   cari_nama('MUHAMMAD REVIANSYAH', 'MCF')")


🔎 FITUR PENCARIAN NAMA
✅ Fungsi cari_nama() siap digunakan!

📖 Cara menggunakan:
   1. cari_nama('NAMA LENGKAP')               → Cari di MCF & MOS
   2. cari_nama('NAMA LENGKAP', 'MCF')        → Cari di MCF saja
   3. cari_nama('NAMA LENGKAP', 'MOS')        → Cari di MOS saja
   4. cari_nama('NAMA LENGKAP', threshold=60) → Threshold lebih rendah

🔍 Contoh:
   cari_nama('LILY FITRI HASANAH')
   cari_nama('MUHAMMAD REVIANSYAH', 'MCF')


In [43]:
cari_nama('yohanes alvin')


🔍 Mencari: yohanes alvin
   Sumber: ALL

--- Mencari di MCF (Azure AI-900) ---
❌ Tidak ditemukan dengan threshold 70
   Skor tertinggi: 0

--- Mencari di MOS (Office 2019) ---
❌ Tidak ditemukan dengan threshold 70
   Skor tertinggi: 0

🔍 Nama-nama yang mirip (skor >= 50):
    70 | YOHANES TRI SUGIARTO (MCF (Azure AI-900))
    70 | YOHANES ANDREAN SIMANJUNTAK (MOS (Office 2019))
    70 | YOHANES KEVIN SINAGA (MOS (Office 2019))
    62 | JOHANNES SARAGIH (MOS (Office 2019))
    62 | YOHANA SANAOU (MOS (Office 2019))

